# 🚀 Enhanced Density Estimation - Complete Pipeline

This notebook demonstrates the full implementation of the Enhanced Density Estimation system.

## Components:
1. **Adaptive KDE**: Volatility-aware kernel density estimation
2. **Regime Detection**: Market regime identification via change points
3. **Tail Risk Estimation**: EVT-based tail modeling with GPD
4. **Integration**: Regime-specific modeling and validation

---

## 📚 Setup and Imports

In [ ]:
# Standard imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Add project to path
sys.path.append(os.path.dirname(os.getcwd()))

# Import our modules
from adaptive_kde import AdaptiveKDE
from regime_detection import MarketRegimeDetector
from tail_risk import TailRiskEstimator
from validation import (
    validate_adaptive_kde, 
    validate_regime_detection, 
    validate_tail_risk,
    cross_validate_models,
    generate_validation_report
)
from utils import (
    load_stock_data,
    calculate_returns,
    plot_density_comparison,
    plot_qq_comparison,
    plot_volatility_scaling,
    plot_regime_risk_profile,
    print_summary_statistics
)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✅ All imports successful!")

## 📊 Load and Prepare Data

In [ ]:
# Load Nifty 50 data
data_path = '../../Data-Working/Nifty-50-Data.csv'
df = load_stock_data(data_path, date_col='Date', price_col='Close')

print(f"📈 Loaded {len(df)} observations")
print(f"📅 Date range: {df.index[0]} to {df.index[-1]}")

# Extract returns
returns = df['Returns'].dropna()

print(f"\n📊 Returns shape: {returns.shape}")
print_summary_statistics(returns, name="Nifty 50")

In [ ]:
# Visualize price and returns
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Price plot
ax1.plot(df.index, df['Price'], color='darkblue', linewidth=1)
ax1.set_ylabel('Price', fontsize=12)
ax1.set_title('Nifty 50: Price and Returns', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Returns plot
ax2.plot(returns.index, returns.values, color='steelblue', linewidth=0.5, alpha=0.7)
ax2.axhline(0, color='red', linestyle='--', linewidth=1)
ax2.set_ylabel('Returns', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🔧 PHASE 1: Adaptive KDE Implementation

In [ ]:
# Initialize and fit Adaptive KDE
print("=" * 80)
print("PHASE 1: ADAPTIVE KDE")
print("=" * 80)

adaptive_kde = AdaptiveKDE(
    returns=returns,
    volatility_window=22,  # ~1 month
    kde_bandwidth='ISJ',  # Improved Sheather-Jones
    boundary_correction=True
)

# Fit the model
adaptive_kde.fit()

# Get statistics
kde_stats = adaptive_kde.get_statistics()
print("\n📊 Model Statistics:")
for key, value in kde_stats.items():
    if isinstance(value, (int, np.integer)):
        print(f"  {key:<30} {value:>15,}")
    elif isinstance(value, (float, np.floating)):
        print(f"  {key:<30} {value:>15.6f}")
    else:
        print(f"  {key:<30} {str(value):>15}")

In [ ]:
# Visualize density estimation
fig = plot_density_comparison(returns, adaptive_kde)
plt.show()

In [ ]:
# Demonstrate volatility scaling
fig = plot_volatility_scaling(adaptive_kde, vol_scenarios=[0.5, 1.0, 1.5, 2.0, 2.5])
plt.show()

In [ ]:
# QQ plots for diagnostics
fig = plot_qq_comparison(returns, adaptive_kde)
plt.show()

## 🎭 PHASE 2: Regime Detection

In [ ]:
print("=" * 80)
print("PHASE 2: REGIME DETECTION")
print("=" * 80)

# Initialize regime detector
regime_detector = MarketRegimeDetector(
    returns=returns,
    volatility_series=adaptive_kde.volatility_series,
    volatility_window=22
)

# Fit using PELT method (automatic breakpoint detection)
regime_detector.fit(method='pelt', penalty=5.0, min_size=30)

print(f"\n✅ Detected {regime_detector.n_regimes} market regimes")
print(f"📍 Breakpoints: {regime_detector.breakpoints}")

In [ ]:
# Display regime statistics
print("\n📊 Regime Characteristics:")
print(regime_detector.regime_stats.to_string())

In [ ]:
# Display regime date ranges
regime_dates = regime_detector.get_regime_dates()
print("\n📅 Regime Date Ranges:")
print(regime_dates.to_string())

In [ ]:
# Visualize regimes
fig = regime_detector.plot_regimes(prices=df['Price'])
plt.show()

## 📉 PHASE 3: Tail Risk Estimation

In [ ]:
print("=" * 80)
print("PHASE 3: TAIL RISK ESTIMATION")
print("=" * 80)

# Initialize tail risk estimator
tail_estimator = TailRiskEstimator(
    returns=returns,
    threshold_quantile=0.95,  # Use top 5% as extremes
    tail='both'  # Model both left (losses) and right (gains) tails
)

# Fit GPD to tails
tail_estimator.fit()

In [ ]:
# Get tail statistics
tail_stats = tail_estimator.get_statistics()

print("\n📊 Tail Risk Statistics:")
print("\nLeft Tail (Losses):")
print(f"  Threshold: {tail_stats['left_threshold']:.6f}")
print(f"  Shape parameter: {tail_stats['left_shape']:.4f}")
print(f"  Scale parameter: {tail_stats['left_scale']:.6f}")
print(f"  VaR 95%: {tail_stats['left_var_95']:.6f}")
print(f"  VaR 99%: {tail_stats['left_var_99']:.6f}")
print(f"  ES 95%: {tail_stats['left_es_95']:.6f}")
print(f"  ES 99%: {tail_stats['left_es_99']:.6f}")

print("\nRight Tail (Gains):")
print(f"  Threshold: {tail_stats['right_threshold']:.6f}")
print(f"  Shape parameter: {tail_stats['right_shape']:.4f}")
print(f"  Scale parameter: {tail_stats['right_scale']:.6f}")
print(f"  VaR 95%: {tail_stats['right_var_95']:.6f}")
print(f"  VaR 99%: {tail_stats['right_var_99']:.6f}")

In [ ]:
# Visualize tail fits
fig = plot_density_comparison(returns, adaptive_kde, tail_estimator=tail_estimator)
plt.show()

In [ ]:
# Tail diagnostics
fig = plot_qq_comparison(returns, adaptive_kde, tail_estimator=tail_estimator)
plt.show()

## 🔗 PHASE 4: Regime-Specific Modeling

In [ ]:
print("=" * 80)
print("PHASE 4: REGIME-SPECIFIC MODELING")
print("=" * 80)

# Build separate models for each regime
models_by_regime = {}

for regime_id in range(regime_detector.n_regimes):
    print(f"\n🔧 Building models for Regime {regime_id}...")
    
    # Extract regime data
    mask = regime_detector.regime_labels == regime_id
    regime_returns = returns[mask]
    
    if len(regime_returns) < 50:
        print(f"   ⚠️  Skipping regime {regime_id}: insufficient data ({len(regime_returns)} obs)")
        continue
    
    # Fit Adaptive KDE for this regime
    regime_kde = AdaptiveKDE(regime_returns, volatility_window=22)
    regime_kde.fit()
    
    # Fit Tail Risk estimator for this regime
    regime_tail = TailRiskEstimator(regime_returns, threshold_quantile=0.95, tail='both')
    regime_tail.fit()
    
    # Get regime statistics
    regime_stats = regime_detector.regime_stats[regime_detector.regime_stats['regime_id'] == regime_id].iloc[0]
    
    # Store models
    models_by_regime[regime_id] = {
        'kde_model': regime_kde,
        'tail_model': regime_tail,
        'regime_stats': regime_stats.to_dict(),
        'n_observations': len(regime_returns)
    }
    
    print(f"   ✅ Regime {regime_id} complete: {len(regime_returns)} observations")

print(f"\n✅ Built models for {len(models_by_regime)} regimes")

In [ ]:
# Compare densities across regimes
fig, axes = plt.subplots(1, len(models_by_regime), figsize=(6*len(models_by_regime), 5))

if len(models_by_regime) == 1:
    axes = [axes]

for idx, (regime_id, models) in enumerate(models_by_regime.items()):
    kde_model = models['kde_model']
    regime_returns = returns[regime_detector.regime_labels == regime_id]
    
    # Plot histogram
    axes[idx].hist(regime_returns, bins=40, density=True, alpha=0.5, 
                  color='skyblue', edgecolor='black', label='Empirical')
    
    # Plot KDE
    x_grid = np.linspace(regime_returns.min() * 1.2, regime_returns.max() * 1.2, 300)
    density = kde_model.pdf(x_grid, current_volatility=kde_model.baseline_volatility)
    axes[idx].plot(x_grid, density, 'r-', linewidth=2, label='Adaptive KDE')
    
    # Labels
    axes[idx].set_xlabel('Returns', fontsize=11)
    axes[idx].set_ylabel('Density', fontsize=11)
    axes[idx].set_title(f'Regime {regime_id}\n(n={len(regime_returns)}, vol={kde_model.baseline_volatility:.4f})',
                       fontsize=12, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive risk profile across regimes
if len(models_by_regime) > 1:
    fig = plot_regime_risk_profile(models_by_regime)
    plt.show()

## ✅ PHASE 5: Validation

In [ ]:
print("=" * 80)
print("PHASE 5: MODEL VALIDATION")
print("=" * 80)

# Split data for validation (80/20)
split_idx = int(len(returns) * 0.8)
returns_train = returns.iloc[:split_idx]
returns_test = returns.iloc[split_idx:]

print(f"\n📊 Train set: {len(returns_train)} observations")
print(f"📊 Test set: {len(returns_test)} observations")

In [ ]:
# Validate Adaptive KDE
kde_validation = validate_adaptive_kde(adaptive_kde, returns_test=returns_test)

print("\n📋 Adaptive KDE Validation Results:")
for key, value in kde_validation.items():
    if isinstance(value, (float, np.floating)):
        print(f"  {key:<30} {value:>12.6f}")
    else:
        print(f"  {key:<30} {str(value):>12}")

In [ ]:
# Validate Regime Detection
regime_validation = validate_regime_detection(regime_detector)

print("\n📋 Regime Detection Validation Results:")
for key, value in regime_validation.items():
    if key == 'cv_by_regime':
        print(f"  {key:<30} {value}")
    elif isinstance(value, (float, np.floating)):
        print(f"  {key:<30} {value:>12.6f}")
    elif isinstance(value, bool):
        print(f"  {key:<30} {'Yes' if value else 'No':>12}")
    else:
        print(f"  {key:<30} {str(value):>12}")

In [ ]:
# Validate Tail Risk
tail_validation = validate_tail_risk(tail_estimator, returns_test=returns_test)

print("\n📋 Tail Risk Validation Results:")
for key, value in tail_validation.items():
    if isinstance(value, (float, np.floating)):
        print(f"  {key:<40} {value:>12.6f}")
    else:
        print(f"  {key:<40} {str(value):>12}")

In [ ]:
# Cross-validation
print("\n🔄 Running cross-validation (this may take a minute)...")
cv_results = cross_validate_models(returns, n_splits=5)

print("\n📋 Cross-Validation Results:")
for key, value in cv_results.items():
    print(f"  {key:<30} {value:>12.6f}")

In [ ]:
# Generate comprehensive validation report
validation_report = generate_validation_report(
    kde_validation,
    regime_validation,
    tail_validation,
    cv_results
)

print(validation_report)

In [ ]:
# Save validation report
report_path = './validation_report.txt'
with open(report_path, 'w') as f:
    f.write(validation_report)
print(f"\n💾 Validation report saved to: {report_path}")

## 💾 Save Results

In [ ]:
# Save regime statistics
regime_detector.regime_stats.to_csv('./regime_statistics.csv', index=False)
print("✅ Saved: regime_statistics.csv")

# Save regime labels with dates
regime_labels_df = pd.DataFrame({
    'Date': returns.index,
    'Returns': returns.values,
    'Regime': regime_detector.regime_labels.values,
    'Volatility': regime_detector.volatility_series.values
})
regime_labels_df.to_csv('./regime_labels.csv', index=False)
print("✅ Saved: regime_labels.csv")

# Save tail risk statistics
tail_stats_df = pd.DataFrame([tail_stats])
tail_stats_df.to_csv('./tail_risk_statistics.csv', index=False)
print("✅ Saved: tail_risk_statistics.csv")

print("\n🎉 All results saved successfully!")

## 🎯 Summary and Next Steps

In [ ]:
print("=" * 80)
print("🎉 ENHANCED DENSITY ESTIMATION - COMPLETE")
print("=" * 80)
print("\n✅ Successfully Implemented:")
print("   1. ✓ Adaptive KDE with volatility normalization")
print("   2. ✓ Market regime detection via change points")
print("   3. ✓ EVT-based tail risk estimation")
print(f"   4. ✓ Regime-specific models ({len(models_by_regime)} regimes)")
print("   5. ✓ Comprehensive validation suite")

print("\n📊 Key Outputs:")
print(f"   • Number of regimes: {regime_detector.n_regimes}")
print(f"   • Volatility range: {regime_detector.regime_stats['volatility'].min():.6f} - {regime_detector.regime_stats['volatility'].max():.6f}")
print(f"   • VaR 99% (full data): {tail_stats['left_var_99']:.6f}")
print(f"   • ES 99% (full data): {tail_stats['left_es_99']:.6f}")

print("\n🚀 Ready for Next Step:")
print("   → Step 4.2: Monte Carlo Simulation")
print("   → Use models_by_regime dictionary for regime-aware simulation")
print("   → Generate realistic price paths with regime transitions")

print("\n💡 Saved Artifacts:")
print("   • regime_statistics.csv")
print("   • regime_labels.csv")
print("   • tail_risk_statistics.csv")
print("   • validation_report.txt")

print("\n" + "=" * 80)